# AutoReserve — Provisionnement technique auto (pipeline complet)

**Stage actuariel — BH Assurance (Tunis)** · *Nesrine El Amri*

Chaine complete : des bases de production jusqu'a la quantification de
l'incertitude. Methodes codees *from scratch* dans le package `autoreserve`.

**Sommaire**
1. Primes & PPNA (unification + validation vs Compte Technique)
2. Segmentation CORPOREL / MATERIEL
3. Les 3 triangles (ouverture, paiements, charge)
4. Chain-Ladder deterministe
5. Ecretement des sinistres graves
6. Bornhuetter-Ferguson (a priori = valeur encadrant 2024)
7. Methode du Cout Moyen
8. Consolidation PSAP / IBNR vs SAP comptable reel
9. Bootstrap ODP — incertitude
10. Synthese & conclusions


In [ ]:
import sys
sys.path.insert(0, r"C:\\Users\\user\\Desktop\\autoreserve")
import numpy as np, pandas as pd
import plotly.io as pio
pio.renderers.default = "notebook_connected"
MDT = 1_000_000
pd.options.display.float_format = lambda v: f"{v:,.0f}"

from autoreserve.config import (PROCESSED_DIR, ANNEE_DEBUT_SINISTRES,
    PRIMES_EMISES_2024_CT, PPNA_2024_CT, PRIME_ACQUISE_2024_CT)
from autoreserve.sinistres.loader import load_sinistres

df = load_sinistres(annee_debut=ANNEE_DEBUT_SINISTRES, verbose=False)   # 2020-2024
df_ppna = pd.read_parquet(PROCESSED_DIR / "base_avec_ppna_v2.parquet")  # base corrigee brute
df_unif = pd.read_parquet(PROCESSED_DIR / "base_unifiee.parquet")
print(f"Sinistres {df['ANNEE_SURVENANCE'].min()}-{df['ANNEE_SURVENANCE'].max()} : {len(df):,} lignes")
print(f"Base unifiee : {len(df_unif):,} polices | base PPNA : {len(df_ppna):,} lignes")



## 1. Primes & PPNA (Etape 1)

Unification de **6 bases** (3 fichiers CA x individuelle/flotte), selection de la
**prime corrigee**, exclusion des **contre-passations** (annulations), puis calcul
de la **PPNA reglementaire (norme CGA)** au 31/12/2024 et rapprochement avec le
**Compte Technique de l'encadrant**.

In [ ]:
d24 = df_ppna[df_ppna["ANNEE_EXERCICE"] == 2024]
primes = pd.to_numeric(d24["TOTAL_PRIME_NETTE"], errors="coerce").sum()
ppna = pd.to_numeric(d24["PPNA_REGLEMENTAIRE"], errors="coerce").sum()
pa = pd.to_numeric(d24["PRIME_ACQUISE_NETTE"], errors="coerce").sum()

rap = pd.DataFrame({
    "Notre calcul": [primes, pa, ppna],
    "Compte Technique": [PRIMES_EMISES_2024_CT, PRIME_ACQUISE_2024_CT, PPNA_2024_CT],
}, index=["Primes emises", "Prime acquise", "PPNA"])
rap["Ecart %"] = (100 * (rap["Notre calcul"] - rap["Compte Technique"]) / rap["Compte Technique"]).round(2)
print(f"Taux de non-acquisition : {100*ppna/primes:.1f}% (cible {100*PPNA_2024_CT/PRIMES_EMISES_2024_CT:.1f}%)")
rap


Les primes emises sont rapprochees a **-0,18 %** et le taux de non-acquisition
reproduit celui du Compte Technique. L'ecart residuel sur la PPNA reflete une
difference de perimetre d'extraction (non reproductible depuis le fichier).

## 2. Segmentation CORPOREL / MATERIEL

Traitement separe : le corporel se developpe lentement et concentre les gros
sinistres ; le materiel se regle vite.

In [ ]:
df["PAYE"] = pd.to_numeric(df["SINISTRE_PAYE"], errors="coerce").fillna(0)
seg = df.groupby("NATURE_SINISTRE").agg(
    nb_sinistres=("NUM_SINISTRE", "nunique"), paye_MDT=("PAYE", lambda s: s.sum()/MDT))
seg["paye_MDT"] = seg["paye_MDT"].round(1)
seg


## 3. Les 3 triangles (base 2020-2024)

Methodologie de l'encadrant : ouverture (nombre), paiements cumules, charge.

In [ ]:
from autoreserve.sinistres.triangles import (
    build_triangle_ouverture, build_triangle_paiements, build_triangle_charge)
tri_ouv = build_triangle_ouverture(df, nature="CORPOREL", cumule=True)
tri_pai = build_triangle_paiements(df, nature="CORPOREL", cumule=True)
tri_chg = build_triangle_charge(df, nature="CORPOREL")
print("Triangle OUVERTURE (nombre) — CORPOREL :"); display(tri_ouv.round(0))
print("Triangle PAIEMENTS cumules (M DT) — CORPOREL :"); display((tri_pai/MDT).round(2))
print("Triangle CHARGE = paye+SAP (M DT) — CORPOREL :"); display((tri_chg/MDT).round(2))


In [ ]:
from autoreserve.reporting.charts import fig_triangle_developpement
fig_triangle_developpement(tri_pai, "CORPOREL")


## 4. Chain-Ladder deterministe

Projection via les coefficients de developpement (valide sur Mack 1993).

In [ ]:
from autoreserve.sinistres.chain_ladder import chain_ladder_full
cl = chain_ladder_full(tri_pai)
print("Facteurs f_k :"); print(cl["link_ratios"].round(3).to_string())
print(f"\nPaye={cl['total_paye']/MDT:,.1f}  Ultime={cl['total_ultime']/MDT:,.1f}"
      f"  Reserves={cl['total_reserves']/MDT:,.1f} M DT")


## 5. Ecretement des sinistres graves

Capping des paiements au seuil ; on raisonne en paiements (le SAP est un stock).

In [ ]:
from autoreserve.sinistres.ecretement import sensibilite_seuil, chain_ladder_ecrete
s = sensibilite_seuil(df, nature="CORPOREL")
aff = s.copy(); aff["seuil"]=(aff["seuil"]/MDT).round(3)
for c in ["ultime_brut","ultime_ecrete","ecart_vs_brut"]: aff[c]=(aff[c]/MDT).round(2)
aff


In [ ]:
from autoreserve.reporting.charts import fig_sensibilite_graves
fig_sensibilite_graves(s, "CORPOREL")


## 6. Bornhuetter-Ferguson

A priori = Prime Acquise x Loss Ratio. La prime acquise 2024 est la **valeur
officielle de l'encadrant** ; le loss ratio est calibre sur les annees a prime
reelle (2022-2023).

In [ ]:
print(f"Prime acquise 2024 (encadrant) utilisee en a priori : {PRIME_ACQUISE_2024_CT:,.0f} DT")

## 7. Methode du Cout Moyen (encadrant)

In [ ]:
from autoreserve.sinistres.methode_cm import methode_cm_encadrant
cm = methode_cm_encadrant(df, nature="CORPOREL")
print(f"CM corporel — charge actuelle={cm['total_charge_actuelle']/MDT:,.1f}"
      f"  tardifs={cm['total_charge_tardifs']/MDT:,.1f}  ultime={cm['total_ultime_cm']/MDT:,.1f} M DT")


## 8. Consolidation PSAP / IBNR vs SAP comptable reel

In [ ]:
from autoreserve.sinistres.consolidation import consolider_best_estimate
conso = consolider_best_estimate(df, df_ppna=df_ppna)
syn = conso["synthese"].copy()
for c in ["reserves_estimees","sap_comptable","ibnr_estime","boni_mali"]: syn[c]=(syn[c]/MDT).round(2)
display(syn[["methode_retenue","reserves_estimees","sap_comptable","ibnr_estime","boni_mali"]])
t=conso["total"]
print(f"BE sinistres={t['best_estimate_sinistres']/MDT:,.1f}M  SAP={t['sap_comptable']/MDT:,.1f}M"
      f"  IBNR={t['ibnr_estime']/MDT:+,.1f}M")


In [ ]:
from autoreserve.reporting.charts import fig_reserves_par_methode, fig_boni_mali
fig_reserves_par_methode(conso["comparatifs_par_segment"]["CORPOREL"], "CORPOREL")


In [ ]:
fig_boni_mali(conso["synthese"])

## 9. Bootstrap ODP — incertitude

In [ ]:
from autoreserve.sinistres.bootstrap import bootstrap_odp
boot = {s: bootstrap_odp(build_triangle_paiements(df, nature=s, cumule=True), n_sim=2000, seed=42)
        for s in ["CORPOREL","MATERIEL"]}
pd.DataFrame({s: {"Best Estimate (M)": r["best_estimate_cl"]/MDT, "SE (M)": r["std"]/MDT,
                  "CV %": r["cv_pct"], "VaR 99,5% (M)": r["quantiles"]["99.5%"]/MDT}
             for s, r in boot.items()}).round(2)


In [ ]:
from autoreserve.reporting.charts import fig_bootstrap_distribution
fig_bootstrap_distribution(boot["CORPOREL"], "CORPOREL")


## 10. Synthese & conclusions

- PPNA **validee** (taux de non-acquisition conforme au Compte Technique ; primes
  rapprochees a -0,18 %).
- Methodes sinistres sur **2020-2024**, 4 methodes comparees au **SAP comptable reel**.
- Pic corporel **ecrete** ; incertitude **quantifiee** (CV corporel >> materiel).
- Choix cles : prime **corrigee brute**, a priori BF = **valeur encadrant 2024**,
  SAP (stock) jamais somme sur le developpement.
